# Using custom containers with Vertex AI Training

**Learning Objectives:**
1. Learn how to create a train and a validation split with BigQuery
1. Learn how to wrap a machine learning model into a Docker container and train in on Vertex AI
1. Learn how to use the hyperparameter tuning engine on Vertex AI to find the best hyperparameters
1. Learn how to deploy a trained machine learning model on Vertex AI as a REST API and query it

In this lab, you develop, package as a docker image, and run on **Vertex AI Training** a training application that trains a multi-class classification model that predicts the type of forest cover from cartographic data. The [dataset](../../../datasets/covertype/README.md) used in the lab is based on **Covertype Data Set** from UCI Machine Learning Repository.

The training code uses `scikit-learn` for data pre-processing and modeling. The code has been instrumented using the `hypertune` package so it can be used with **Vertex AI** hyperparameter tuning.


In [1]:
import os
import time

import pandas as pd
from google.cloud import aiplatform, bigquery
from google.cloud.aiplatform import hyperparameter_tuning as hpt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

%load_ext google.cloud.bigquery

/home/jupyter/asl-ml-immersion/asl_mlops/.venv/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


The google.cloud.bigquery extension is already loaded. To reload it, use:
  %reload_ext google.cloud.bigquery


## Configure environment settings

Set location paths, connections strings, and other environment settings. Make sure to update   `REGION`, and `ARTIFACT_STORE`  with the settings reflecting your lab environment. 

- `REGION` - the compute region for Vertex AI Training and Prediction
- `ARTIFACT_STORE` - A GCS bucket in the created in the same region.

In [2]:
REGION = "us-central1"

PROJECT_ID = !(gcloud config get-value core/project)
PROJECT_ID = PROJECT_ID[0]

ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"

DATA_ROOT = f"{ARTIFACT_STORE}/data"
JOB_DIR_ROOT = f"{ARTIFACT_STORE}/jobs"
TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"
API_ENDPOINT = f"{REGION}-aiplatform.googleapis.com"

In [3]:
os.environ["JOB_DIR_ROOT"] = JOB_DIR_ROOT
os.environ["TRAINING_FILE_PATH"] = TRAINING_FILE_PATH
os.environ["VALIDATION_FILE_PATH"] = VALIDATION_FILE_PATH
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION

We now create the `ARTIFACT_STORE` bucket if it's not there. Note that this bucket should be created in the region specified in the variable `REGION` (if you have already a bucket with this name in a different region than `REGION`, you may want to change the `ARTIFACT_STORE` name so that you can recreate a bucket in `REGION` with the command in the cell below).

In [4]:
!gsutil ls | grep ^{ARTIFACT_STORE}/$ || gsutil mb -l {REGION} {ARTIFACT_STORE}

Creating gs://qwiklabs-asl-00-8534302ca246-kfp-artifact-store/...


## Importing the dataset into BigQuery

In [5]:
%%bash

DATASET_LOCATION=US
DATASET_ID=covertype_dataset
TABLE_ID=covertype
DATA_SOURCE=gs://asl-public/data/covertype/dataset.csv
SCHEMA=Elevation:INTEGER,\
Aspect:INTEGER,\
Slope:INTEGER,\
Horizontal_Distance_To_Hydrology:INTEGER,\
Vertical_Distance_To_Hydrology:INTEGER,\
Horizontal_Distance_To_Roadways:INTEGER,\
Hillshade_9am:INTEGER,\
Hillshade_Noon:INTEGER,\
Hillshade_3pm:INTEGER,\
Horizontal_Distance_To_Fire_Points:INTEGER,\
Wilderness_Area:STRING,\
Soil_Type:STRING,\
Cover_Type:INTEGER

bq --location=$DATASET_LOCATION --project_id=$PROJECT_ID mk --dataset $DATASET_ID

bq --project_id=$PROJECT_ID --dataset_id=$DATASET_ID load \
--source_format=CSV \
--skip_leading_rows=1 \
--replace \
$TABLE_ID \
$DATA_SOURCE \
$SCHEMA

Dataset 'qwiklabs-asl-00-8534302ca246:covertype_dataset' successfully created.


Waiting on bqjob_r48617b029d543f89_0000019c9ca7b6b8_1 ... (2s) Current status: DONE   


## Explore the Covertype dataset 

In [6]:
%%bigquery
SELECT *
FROM `covertype_dataset.covertype`

Query is running:   0%|          |

Downloading:   0%|          |

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area,Soil_Type,Cover_Type
0,2559,0,0,510,16,1113,218,238,156,1332,Commanche,C2703,2
1,2395,0,0,60,6,1170,218,238,156,1054,Cache,C2717,2
2,2916,0,0,268,3,4948,218,238,156,2799,Rawah,C3502,0
3,2283,0,33,458,126,626,154,160,129,618,Cache,C4703,2
4,2231,0,35,150,87,1224,147,151,125,1426,Cache,C4703,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,2217,354,32,95,35,811,150,166,142,1082,Cache,C4703,5
99996,2418,354,32,90,53,684,150,167,142,1368,Cache,C4703,1
99997,2263,354,32,216,152,674,149,165,141,218,Cache,C4703,2
99998,2699,356,32,42,19,2654,154,167,138,6662,Rawah,C7745,1


## Create training and validation splits

Use BigQuery to sample training and validation splits and save them to GCS storage
### Create a training split

In [7]:
!bq query \
-n 0 \
--destination_table covertype_dataset.training \
--replace \
--use_legacy_sql=false \
'SELECT * \
FROM `covertype_dataset.covertype` AS cover \
WHERE \
MOD(ABS(FARM_FINGERPRINT(TO_JSON_STRING(cover))), 10) IN (1, 2, 3, 4)' 

Waiting on bqjob_r26bf56fe0a0ce350_0000019c9ca7e36f_1 ... (1s) Current status: DONE   


In [8]:
!bq extract \
--destination_format CSV \
covertype_dataset.training \
$TRAINING_FILE_PATH

Waiting on bqjob_r4a0403165c4f6b70_0000019c9ca7fb0f_1 ... (1s) Current status: DONE   


### Create a validation split

In [9]:
!bq query \
-n 0 \
--destination_table covertype_dataset.validation \
--replace \
--use_legacy_sql=false \
'SELECT * \
FROM `covertype_dataset.covertype` AS cover \
WHERE \
MOD(ABS(FARM_FINGERPRINT(TO_JSON_STRING(cover))), 10) IN (8)' 

Waiting on bqjob_r385e8a6075267eac_0000019c9ca81348_1 ... (1s) Current status: DONE   


In [10]:
!bq extract \
--destination_format CSV \
covertype_dataset.validation \
$VALIDATION_FILE_PATH

Waiting on bqjob_re7d8a583e4eeb69_0000019c9ca829e2_1 ... (0s) Current status: DONE   


In [11]:
df_train = pd.read_csv(TRAINING_FILE_PATH)
df_validation = pd.read_csv(VALIDATION_FILE_PATH)
print(df_train.shape)
print(df_validation.shape)

(40009, 13)
(9836, 13)


## Develop a training application

### Configure the `sklearn` training pipeline.

The training pipeline preprocesses data by standardizing all numeric features using `sklearn.preprocessing.StandardScaler` and encoding all categorical features using `sklearn.preprocessing.OneHotEncoder`. It uses stochastic gradient descent linear classifier (`SGDClassifier`) for modeling.

In [12]:
numeric_feature_indexes = slice(0, 10)
categorical_feature_indexes = slice(10, 12)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_feature_indexes),
        ("cat", OneHotEncoder(), categorical_feature_indexes),
    ]
)

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", SGDClassifier(loss="log_loss", tol=1e-3)),
    ]
)

### Convert all numeric features to `float64`

To avoid warning messages from `StandardScaler` all numeric features are converted to `float64`.

In [13]:
num_features_type_map = {
    feature: "float64" for feature in df_train.columns[numeric_feature_indexes]
}

df_train = df_train.astype(num_features_type_map)
df_validation = df_validation.astype(num_features_type_map)

### Run the pipeline locally.

In [14]:
X_train = df_train.drop("Cover_Type", axis=1)
y_train = df_train["Cover_Type"]
X_validation = df_validation.drop("Cover_Type", axis=1)
y_validation = df_validation["Cover_Type"]

pipeline.set_params(classifier__alpha=0.001, classifier__max_iter=200)
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Calculate the trained model's accuracy.

In [15]:
accuracy = pipeline.score(X_validation, y_validation)
print(accuracy)

0.6929646197641317


### Prepare the hyperparameter tuning application.
Since the training run on this dataset is computationally expensive you can benefit from running a distributed hyperparameter tuning job on Vertex AI Training.

In [16]:
TRAINING_APP_FOLDER = "training_app"
os.makedirs(TRAINING_APP_FOLDER, exist_ok=True)

### Write the tuning script. 

Notice the use of the `hypertune` package to report the `accuracy` optimization metric to Vertex AI hyperparameter tuning service.

In [17]:
%%writefile {TRAINING_APP_FOLDER}/train.py
import os
import subprocess
import sys

import fire
import hypertune
import numpy as np
import pandas as pd
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def train_evaluate(job_dir, training_dataset_path, validation_dataset_path, alpha, max_iter, hptune):
    
    df_train = pd.read_csv(training_dataset_path)
    df_validation = pd.read_csv(validation_dataset_path)

    if not hptune:
        df_train = pd.concat([df_train, df_validation])

    numeric_feature_indexes = slice(0, 10)
    categorical_feature_indexes = slice(10, 12)

    preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_feature_indexes),
        ('cat', OneHotEncoder(), categorical_feature_indexes) 
    ])

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SGDClassifier(loss='log_loss',tol=1e-3))
    ])

    num_features_type_map = {feature: 'float64' for feature in df_train.columns[numeric_feature_indexes]}
    df_train = df_train.astype(num_features_type_map)
    df_validation = df_validation.astype(num_features_type_map) 

    print('Starting training: alpha={}, max_iter={}'.format(alpha, max_iter))
    X_train = df_train.drop('Cover_Type', axis=1)
    y_train = df_train['Cover_Type']

    pipeline.set_params(classifier__alpha=alpha, classifier__max_iter=int(max_iter))
    pipeline.fit(X_train, y_train)

    if hptune:
        X_validation = df_validation.drop('Cover_Type', axis=1)
        y_validation = df_validation['Cover_Type']
        accuracy = pipeline.score(X_validation, y_validation)
        print('Model accuracy: {}'.format(accuracy))
        # Log it with hypertune
        hpt = hypertune.HyperTune()
        hpt.report_hyperparameter_tuning_metric(
          hyperparameter_metric_tag='accuracy',
          metric_value=accuracy
        )

    # Save the model
    if not hptune:
        model_filename = 'model.pkl'
        with open(model_filename, 'wb') as model_file:
            pickle.dump(pipeline, model_file)
        gcs_model_path = "{}/{}".format(job_dir, model_filename)
        subprocess.check_call(['gsutil', 'cp', model_filename, gcs_model_path], stderr=sys.stdout)
        print("Saved model in: {}".format(gcs_model_path)) 
    
if __name__ == "__main__":
    fire.Fire(train_evaluate)

Writing training_app/train.py


### Package the script into a docker image.

Notice that we are installing specific versions of `scikit-learn` and `pandas` in the training image. This is done to make sure that the training runtime in the training container is aligned with the serving runtime in the serving container. 

Make sure to update the URI for the base image so that it points to your project's **Artifact Registry**.

In [18]:
%%writefile {TRAINING_APP_FOLDER}/Dockerfile

FROM us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-0
RUN pip install -U fire cloudml-hypertune scikit-learn==1.2.2
WORKDIR /app
COPY train.py .

ENTRYPOINT ["python", "train.py"]

Writing training_app/Dockerfile


### Build the docker image. 

You use **Cloud Build** to build the image and push it your project's **Artifact Registry**. As you use the remote cloud service to build the image, you don't need a local installation of Docker.

In [19]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "trainer_image"
IMAGE_TAG = "latest"
IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{IMAGE_TAG}"

os.environ["IMAGE_URI"] = IMAGE_URI

In [20]:
!gcloud builds submit --tag $IMAGE_URI $TRAINING_APP_FOLDER

Creating temporary archive of 2 file(s) totalling 2.6 KiB before compression.
Uploading tarball of [training_app] to [gs://qwiklabs-asl-00-8534302ca246_cloudbuild/source/1772154797.871605-234945dc596c4a5caa7270685b701718.tgz]
Created [https://cloudbuild.googleapis.com/v1/projects/qwiklabs-asl-00-8534302ca246/locations/global/builds/22b24df5-7a0f-44f3-bbb7-4386dd0a5d22].
Logs are available at [ https://console.cloud.google.com/cloud-build/builds/22b24df5-7a0f-44f3-bbb7-4386dd0a5d22?project=998865242242 ].
Waiting for build to complete. Polling interval: 1 second(s).
----------------------------- REMOTE BUILD OUTPUT ------------------------------
starting build "22b24df5-7a0f-44f3-bbb7-4386dd0a5d22"

FETCHSOURCE
Fetching storage object: gs://qwiklabs-asl-00-8534302ca246_cloudbuild/source/1772154797.871605-234945dc596c4a5caa7270685b701718.tgz#1772154798771890
Copying gs://qwiklabs-asl-00-8534302ca246_cloudbuild/source/1772154797.871605-234945dc596c4a5caa7270685b701718.tgz#1772154798771890

## Submit an Vertex AI hyperparameter tuning job

### Create the hyperparameter configuration file. 
Recall that the training code uses `SGDClassifier`. The training application has been designed to accept two hyperparameters that control `SGDClassifier`:
- Max iterations
- Alpha

The file below configures Vertex AI hypertuning to run up to 5 trials in parallel and to choose from two discrete values of `max_iter` and the linear range between `1.0e-4` and `1.0e-1` for `alpha`.

In [21]:
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"forestcover_tuning_{TIMESTAMP}"
JOB_DIR = f"{JOB_DIR_ROOT}/{JOB_NAME}"

args = [
    f"--job_dir={JOB_DIR}",
    f"--training_dataset_path={TRAINING_FILE_PATH}",
    f"--validation_dataset_path={VALIDATION_FILE_PATH}",
    f"--hptune",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": IMAGE_URI,
            "args": args,
        },
    }
]

custom_job = aiplatform.CustomJob(
    display_name=JOB_NAME,
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{JOB_DIR}/staging",
)

In [ ]:
hpt_job = aiplatform.HyperparameterTuningJob(
    display_name=JOB_NAME,
    custom_job=custom_job,
    metric_spec={
        "accuracy": "maximize",
    },
    parameter_spec={
        "alpha": hpt.DoubleParameterSpec(min=1.0e-4, max=1.0e-1, scale="log"),
        "max_iter": hpt.DiscreteParameterSpec(values=[10, 20], scale="linear"),
    },
    max_trial_count=5,
    parallel_trial_count=5,
)

hpt_job.run()

Creating HyperparameterTuningJob
HyperparameterTuningJob created. Resource name: projects/998865242242/locations/us-central1/hyperparameterTuningJobs/4278069915404992512
To use this HyperparameterTuningJob in another session:
hpt_job = aiplatform.HyperparameterTuningJob.get('projects/998865242242/locations/us-central1/hyperparameterTuningJobs/4278069915404992512')
View HyperparameterTuningJob:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/4278069915404992512?project=998865242242
HyperparameterTuningJob projects/998865242242/locations/us-central1/hyperparameterTuningJobs/4278069915404992512 current state:
3
HyperparameterTuningJob projects/998865242242/locations/us-central1/hyperparameterTuningJobs/4278069915404992512 current state:
3
HyperparameterTuningJob projects/998865242242/locations/us-central1/hyperparameterTuningJobs/4278069915404992512 current state:
3
HyperparameterTuningJob projects/998865242242/locations/us-central1/hyperparameterTuningJobs/427

Go to the Vertex AI Training dashboard and view the progression of the HP tuning job under "Hyperparameter Tuning Jobs".

**Note**: Here is the equivalent gcloud ai command where you can provide the config in yaml file for reference.


```bash
MACHINE_TYPE="n1-standard-4"
REPLICA_COUNT=1
CONFIG_YAML=config.yaml

cat <<EOF > $CONFIG_YAML
studySpec:
  metrics:
  - metricId: accuracy
    goal: MAXIMIZE
  parameters:
  - parameterId: max_iter
    discreteValueSpec:
      values:
      - 10
      - 20
  - parameterId: alpha
    doubleValueSpec:
      minValue: 1.0e-4
      maxValue: 1.0e-1
    scaleType: UNIT_LINEAR_SCALE
  algorithm: ALGORITHM_UNSPECIFIED # results in Bayesian optimization
trialJobSpec:
  workerPoolSpecs:  
  - machineSpec:
      machineType: $MACHINE_TYPE
    replicaCount: $REPLICA_COUNT
    containerSpec:
      imageUri: $IMAGE_URI
      args:
      - --job_dir=$JOB_DIR
      - --training_dataset_path=$TRAINING_FILE_PATH
      - --validation_dataset_path=$VALIDATION_FILE_PATH
      - --hptune
EOF

gcloud ai hp-tuning-jobs create \
    --region=$REGION \
    --display-name=$JOB_NAME \
    --config=$CONFIG_YAML \
    --max-trial-count=5 \
    --parallel-trial-count=5

echo "JOB_NAME: $JOB_NAME"
```

### Retrieve HP-tuning results.

After the job completes you can review the results using GCP Console or programmatically using the following functions (note that this code supposes that the metrics that the hyperparameter tuning engine optimizes is maximized): 

In [ ]:
def get_best_hyperparameters(hpt_job):
    metrics = [
        trial.final_measurement.metrics[0].value for trial in hpt_job.trials
    ]
    best_trial = hpt_job.trials[metrics.index(max(metrics))]
    best_parameters = {
        param.parameter_id: param.value for param in best_trial.parameters
    }
    return best_parameters["alpha"], best_parameters["max_iter"]


alpha, max_iter = get_best_hyperparameters(hpt_job)
print(f"best alpha: {alpha}, best max_iter: {max_iter}")

best alpha: 0.0006041839592710345, best max_iter: 10.0


## Retrain the model with the best hyperparameters

You can now retrain the model using the best hyperparameters and using combined training and validation splits as a training dataset.

### Configure and run the training job

In [ ]:
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"forestcover_training_{TIMESTAMP}"
JOB_DIR = f"{JOB_DIR_ROOT}/{JOB_NAME}"

args = [
    f"--job_dir={JOB_DIR}",
    f"--training_dataset_path={TRAINING_FILE_PATH}",
    f"--validation_dataset_path={VALIDATION_FILE_PATH}",
    f"--alpha={alpha}",
    f"--max_iter={max_iter}",
    f"--nohptune",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": IMAGE_URI,
            "args": args,
        },
    }
]

custom_job = aiplatform.CustomJob(
    display_name=JOB_NAME,
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{JOB_DIR}/staging",
)

custom_job.run()

Creating CustomJob
CustomJob created. Resource name: projects/998865242242/locations/us-central1/customJobs/4659468508847931392
To use this CustomJob in another session:
custom_job = aiplatform.CustomJob.get('projects/998865242242/locations/us-central1/customJobs/4659468508847931392')
View Custom Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/4659468508847931392?project=998865242242
CustomJob projects/998865242242/locations/us-central1/customJobs/4659468508847931392 current state:
2
CustomJob projects/998865242242/locations/us-central1/customJobs/4659468508847931392 current state:
2
CustomJob projects/998865242242/locations/us-central1/customJobs/4659468508847931392 current state:
2
CustomJob projects/998865242242/locations/us-central1/customJobs/4659468508847931392 current state:
2
CustomJob projects/998865242242/locations/us-central1/customJobs/4659468508847931392 current state:
2
CustomJob projects/998865242242/locations/us-central1/customJobs/46594

**Note**: Here is the equivalent bash command to run container training for reference.

Set up env variable in a Python cell.
```python
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
os.environ["alpha"] = alpha
os.environ["max_iter"] = max_iter
os.environ["TIMESTAMP"] = TIMESTAMP
```

Run bash command below

```bash
JOB_NAME=forestcover_training_$TIMESTAMP
JOB_DIR=$JOB_DIR_ROOT/$JOB_NAME

MACHINE_TYPE=n1-standard-4
REPLICA_COUNT=1

WORKER_POOL_SPEC="machine-type=$MACHINE_TYPE,\
replica-count=$REPLICA_COUNT,\
container-image-uri=$IMAGE_URI"

ARGS="--job_dir=$JOB_DIR,\
--training_dataset_path=$TRAINING_FILE_PATH,\
--validation_dataset_path=$VALIDATION_FILE_PATH,\
--alpha=$alpha,\
--max_iter=$max_iter,\
--nohptune"

gcloud ai custom-jobs create \
  --region=$REGION \
  --display-name=$JOB_NAME \
  --worker-pool-spec=$WORKER_POOL_SPEC \
  --args=$ARGS
```

### Examine the training output

The training script saved the trained model as the 'model.pkl' in the `JOB_DIR` folder on GCS.

In [ ]:
!gsutil ls $JOB_DIR

gs://qwiklabs-asl-00-8534302ca246-kfp-artifact-store/jobs/forestcover_training_20260227_012546/model.pkl


## Deploy the model to Vertex AI Prediction

In [ ]:
MODEL_NAME = "forest_cover_classifier"
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
)
SERVING_MACHINE_TYPE = "n1-standard-2"

### Uploading the trained model

In [ ]:
uploaded_model = aiplatform.Model.upload(
    display_name=MODEL_NAME,
    artifact_uri=JOB_DIR,
    serving_container_image_uri=SERVING_CONTAINER_IMAGE_URI,
)

Creating Model
Create Model backing LRO: projects/998865242242/locations/us-central1/models/2764698898298568704/operations/6337083791227486208
Model created. Resource name: projects/998865242242/locations/us-central1/models/2764698898298568704@1
To use this Model in another session:
model = aiplatform.Model('projects/998865242242/locations/us-central1/models/2764698898298568704@1')


### Deploying the uploaded model

In [ ]:
endpoint = uploaded_model.deploy(
    machine_type=SERVING_MACHINE_TYPE,
    accelerator_type=None,
    accelerator_count=None,
)

Creating Endpoint
Create Endpoint backing LRO: projects/998865242242/locations/us-central1/endpoints/8261291116213567488/operations/6997987036544106496
Endpoint created. Resource name: projects/998865242242/locations/us-central1/endpoints/8261291116213567488
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/998865242242/locations/us-central1/endpoints/8261291116213567488')
Deploying model to Endpoint : projects/998865242242/locations/us-central1/endpoints/8261291116213567488
Deploy Endpoint model backing LRO: projects/998865242242/locations/us-central1/endpoints/8261291116213567488/operations/4566043237764038656
Endpoint model deployed. Resource name: projects/998865242242/locations/us-central1/endpoints/8261291116213567488


### Serve predictions
#### Prepare the input file with JSON formated instances.

In [ ]:
instance = [
    2841.0,
    45.0,
    0.0,
    644.0,
    282.0,
    1376.0,
    218.0,
    237.0,
    156.0,
    1003.0,
    "Commanche",
    "C4758",
]
endpoint.predict([instance])

Prediction(predictions=[1.0], deployed_model_id='8949972423294320640', metadata=None, model_version_id='1', model_resource_name='projects/998865242242/locations/us-central1/models/2764698898298568704', explanations=None)

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.